# pymumo
Python API for reading `.mmeaf` files produced by mumo.

In [1]:
# Run once to install in your venv
# !uv pip install -e .
# !uv pip install pandas

In [2]:
from mumo import MumoDoc, patterns_df, gaps, pauses, overlaps_by_timing

## Loading a file

In [3]:
doc = MumoDoc('./tests/fixtures/test.mmeaf')
doc

## Utterances

`doc.utterances` returns all utterance blocks in transcript order.

In [4]:
for utt in doc.utterances:
    ts = f'{utt.start_time:.3f}–{utt.end_time:.3f}s' if utt.start_time is not None else 'no time'
    print(f'[{utt.participant}] ({ts}) {utt.text}')

[SHA] (0.000–1.000s) ugliest so we gotta make c'ncessions hh ˙hhhheh
[VIV] (1.282–2.282s) n:heyuh
[MIC] (2.467–3.467s) I do' wan't 'day
[VIV] (3.645–4.645s) Move a little, can you? 
[MIC] (4.872–5.084s) Mostly taragoose
[VIV] (5.327–6.327s) °Thanks°
[MIC] (6.734–7.734s) Bes' girl does'n it?
[SHA] (9.820–10.820s) hm-hmh yih hmh (.) ˙huhh Best nh hnh Best


In [5]:
utt = doc.utterances[0]

print('id:         ', utt.id)
print('participant:', utt.participant)
print('start_time: ', utt.start_time)
print('end_time:   ', utt.end_time)
print('text:       ', utt.text)

id:          1783311184358-1k111g6m451n
participant: SHA
start_time:  0.0
end_time:    1.0
text:        ugliest so we gotta make c'ncessions hh ˙hhhheh


## Tokens

`utt.tokens` — all tokens including whitespace.  
`utt.words` — non-whitespace tokens (words, punctuation, gaps).

In [6]:
utt = doc.utterances[0]

print('words:', [t.text for t in utt.words])
print('kinds:', [(t.text, t.kind) for t in utt.tokens])

words: ['ugliest', 'so', 'we', 'gotta', 'make', "c'ncessions", 'hh', '˙hhhheh']
kinds: [('ugliest', 'word'), (' ', 'ws'), ('so', 'word'), (' ', 'ws'), ('we', 'word'), (' ', 'ws'), ('gotta', 'word'), (' ', 'ws'), ('make', 'word'), (' ', 'ws'), ("c'ncessions", 'word'), (' ', 'ws'), ('hh', 'word'), (' ', 'ws'), ('˙hhhheh', 'word')]


In [7]:
# Every token links back to its utterance
tok = utt.words[0]
print(tok)
print('utterance:', tok.utterance)

Token(word, 'ugliest')
utterance: Utterance('SHA', "ugliest so we gotta make c'ncessions hh ")


## Continuations

When an utterance is a continuation of another (split-turn / latching), you can navigate the chain.

In [8]:
for utt in doc.utterances:
    if utt.is_continuation:
        print(f'{utt!r} continues {utt.head!r}')
        print('  full chain:', utt.chain)

# Show continuations from the head side
for utt in doc.utterances:
    if utt.continuations:
        print(f'{utt!r} has continuations: {utt.continuations}')

## Overlaps

Overlap brackets in CA notation mark where speakers' speech is simultaneous.
Each `mm:inline_mark type="overlap_bracket"` on an utterance carries a `group_id`
that ties together the utterances involved.

**Per-utterance marks** tell you where (by character offset) the overlap starts/ends
in that utterance's text.  
**Overlap groups** aggregate all marks with the same `group_id` across the whole document.

In [9]:
# Overlap marks on individual utterances
for utt in doc.utterances:
    if utt.overlap_marks:
        print(f'[{utt.participant}] {utt.text!r}')
        for mark in utt.overlap_marks:
            snippet = utt.text[mark.char_offset:mark.char_offset + 10]
            print(f'  {mark.kind:5} group={mark.group_id!r}  offset={mark.char_offset}  → {snippet!r}...')

[SHA] "ugliest so we gotta make c'ncessions hh ˙hhhheh"
  start group='a'  offset=16  → 'tta make c'...
  end   group='a'  offset=32  → 'ions hh ˙h'...
  start group='b'  offset=44  → 'heh'...
[VIV] 'n:heyuh'
  start group='a'  offset=0  → 'n:heyuh'...
  end   group='a'  offset=7  → ''...
[MIC] "I do' wan't 'day"
  start group='b'  offset=0  → "I do' wan'"...
[VIV] 'Move a little, can you? '
  start group='c'  offset=15  → 'can you? '...
  end   group='c'  offset=24  → ''...
[MIC] 'Mostly taragoose'
  start group='c'  offset=0  → 'Mostly tar'...
  end   group='c'  offset=9  → 'ragoose'...
[MIC] "Bes' girl does'n it?"
  start group='d'  offset=17  → 'it?'...


In [10]:
# All overlap groups in the document
for og in doc.overlap_groups:
    print(og)
    for utt in og.utterances:
        marks = [m for m in utt.overlap_marks if m.group_id == og.id]
        offsets = ', '.join(f'{m.kind}@{m.char_offset}' for m in marks)
        print(f'  [{utt.participant}] {offsets}  text={utt.text!r}')

OverlapGroup('a', [SHA, VIV])
  [SHA] start@16, end@32  text="ugliest so we gotta make c'ncessions hh ˙hhhheh"
  [VIV] start@0, end@7  text='n:heyuh'
OverlapGroup('b', [SHA, MIC])
  [SHA] start@44  text="ugliest so we gotta make c'ncessions hh ˙hhhheh"
  [MIC] start@0  text="I do' wan't 'day"
OverlapGroup('c', [VIV, MIC])
  [VIV] start@15, end@24  text='Move a little, can you? '
  [MIC] start@0, end@9  text='Mostly taragoose'
OverlapGroup('d', [MIC])
  [MIC] start@17  text="Bes' girl does'n it?"


In [11]:
# Overlap group timing: intersection of the involved utterances' time ranges.
# start_time > end_time means EAF timestamps don't confirm the bracket overlap
# (transcriber perception vs. automated alignment).
for og in doc.overlap_groups:
    print(f'{og}  t={og.start_time:.3f}–{og.end_time:.3f}s')

OverlapGroup('a', [SHA, VIV])  t=1.282–1.000s
OverlapGroup('b', [SHA, MIC])  t=2.467–1.000s
OverlapGroup('c', [VIV, MIC])  t=4.872–4.645s
OverlapGroup('d', [MIC])  t=6.734–7.734s


In [12]:
# Look up a single group by id
og = doc.overlap_group('a')
print(og)
print('utterances:', og.utterances)

OverlapGroup('a', [SHA, VIV])
utterances: [Utterance('SHA', "ugliest so we gotta make c'ncessions hh "), Utterance('VIV', 'n:heyuh')]


## Textlets

Textlets are annotated sub-spans of an utterance.

In [13]:
for tl in doc.textlets:
    print(tl)
    print('  text:      ', repr(tl.text))
    print('  utterance: ', tl.utterance)
    print('  char span: ', tl.start, '–', tl.end)
    print('  patterns:  ', tl.patterns)

Textlet('°Thanks°')
  text:       '°Thanks°'
  utterance:  Utterance('VIV', '°Thanks°')
  char span:  0 – 8
  patterns:   []


## Patterns

Patterns are instances of a pattern schema (e.g. Adjacency Pair, Repair).
Slots are the filled positions within a pattern.

In [14]:
for pattern in doc.patterns:
    print(pattern)
    print('  schema:', pattern.schema.name)
    print('  note:  ', pattern.note)
    for slot in pattern:
        print(f'  [{slot.name}] anchor_kind={slot.anchor_kind}')
        print(f'         text={slot.text!r}')
        print(f'         utterance={slot.utterance}')
        for mv in slot.metrics:
            print(f'         metric {mv.name!r} = {mv.value!r}')

Pattern('Adjacency Pair', 2 slot(s))
  schema: Adjacency Pair
  note:   None
  [fpp] anchor_kind=utterance
         text="Bes' girl does'n it?"
         utterance=Utterance('MIC', "Bes' girl does'n it?")
         metric 'FPP type' = 'assessment'
  [spp] anchor_kind=utterance
         text=None
         utterance=None
         metric 'SPP type' = 'acceptance'


In [15]:
# Access a slot by name
pattern = doc.patterns[0]
fpp = pattern['fpp']

print('FPP utterance:', fpp.utterance)
print('FPP text:     ', fpp.text)
print('FPP type:     ', fpp['FPP type'])   # metric by name

FPP utterance: Utterance('MIC', "Bes' girl does'n it?")
FPP text:      Bes' girl does'n it?
FPP type:      assessment


In [16]:
# The anchor gives you a typed, structured object
for slot in pattern:
    print(f'{slot.name}: {slot.anchor}')

fpp: UtteranceAnchor(kind='utterance', utterance=Utterance('MIC', "Bes' girl does'n it?"))
spp: None


## Universal lookup

`doc[id]` resolves any ID — utterance, textlet, or pattern.

In [17]:
utt_id = doc.utterances[0].id
print(doc[utt_id])

if doc.textlets:
    tl_id = doc.textlets[0].id
    print(doc[tl_id])

if doc.patterns:
    p_id = doc.patterns[0].id
    print(doc[p_id])

Utterance('SHA', "ugliest so we gotta make c'ncessions hh ")
Textlet('°Thanks°')
Pattern('Adjacency Pair', 2 slot(s))


## Pattern schemas

Inspect what schemas are defined in the file and what slots they contain.

In [18]:
for schema in doc.pattern_schemas:
    print(f'{schema.name}  ({schema.description})')
    for slot in schema.slots:
        required = '(required)' if slot.required else '(optional)'
        print(f'  {slot.name!r:20} anchor={slot.anchor_kind}  {required}')
        for m in slot.metrics:
            print(f'    metric: {m.name!r}  type={m.type}')

Repair  (Other-initiated self-repair (Schegloff et al. 1977))
  'trouble'            anchor=textlet  (required)
    metric: 'Trouble type'  type=categorical
  'initiation'         anchor=textlet  (required)
    metric: 'Initiation type'  type=categorical
  'solution'           anchor=textlet  (optional)
    metric: 'Resolved?'  type=boolean
    metric: 'Solution type'  type=categorical
Adjacency Pair  (First and second pair parts (Schegloff & Sacks 1973))
  'fpp'                anchor=utterance  (required)
    metric: 'FPP type'  type=categorical
  'spp'                anchor=utterance  (required)
    metric: 'SPP type'  type=categorical
    metric: 'Preferred?'  type=boolean


## EAF annotations

All standard ELAN tiers and annotations are accessible alongside mumo data.

In [19]:
# Tiers available in the file
print(list(doc.tiers.keys()))

['utt:SHA', 'utt:VIV', 'utt:MIC', 'SHA', 'POS']


In [20]:
# All annotations on the POS tier
for ann in doc.eaf_tier('POS'):
    print(f'{ann.id}: {ann.value!r}  parent={ann.parent}')

a41: 'ADJ'  parent=EafAnnotation('SHA', 'ugliest')
a42: 'CONJ'  parent=EafAnnotation('SHA', 'so')
a43: 'NOUN'  parent=EafAnnotation('SHA', 'we')
a44: 'VERB'  parent=EafAnnotation('SHA', 'gotta')
a45: 'VERB'  parent=EafAnnotation('SHA', 'make')
a46: 'NOUN'  parent=EafAnnotation('SHA', "c'ncessions")
a47: 'BREATH'  parent=EafAnnotation('SHA', 'hh')
a48: 'BREATH'  parent=EafAnnotation('SHA', '˙hhhheh')
a49: ''  parent=EafAnnotation('SHA', 'hm-hmh')
a50: ''  parent=EafAnnotation('SHA', 'yih')
a51: ''  parent=EafAnnotation('SHA', 'hmh')
a52: ''  parent=EafAnnotation('SHA', '(.)')
a53: ''  parent=EafAnnotation('SHA', '˙huhh')
a54: ''  parent=EafAnnotation('SHA', 'Best')
a55: ''  parent=EafAnnotation('SHA', 'nh')
a56: ''  parent=EafAnnotation('SHA', 'hnh')
a57: ''  parent=EafAnnotation('SHA', 'Best')


In [21]:
# Query by time range (in seconds)
overlapping = doc.annotations_overlapping(0.0, 2.0)
for ann in overlapping:
    print(ann, ann.time)

EafAnnotation('utt:SHA', "ugliest so we gotta make c'ncessions hh ˙hhhheh") (0.0, 1.0)
EafAnnotation('utt:VIV', 'n:heyuh') (1.282, 2.282)


In [22]:
# Each utterance links to its EAF annotation
for utt in doc.utterances:
    eaf = utt.eaf_annotation
    if eaf:
        print(f'{utt.participant}: eaf_id={eaf.id}  value={eaf.value!r}')

SHA: eaf_id=a1  value="ugliest so we gotta make c'ncessions hh ˙hhhheh"
VIV: eaf_id=a3  value='n:heyuh'
MIC: eaf_id=a6  value="I do' wan't 'day"
VIV: eaf_id=a4  value='Move a little, can you? '
MIC: eaf_id=a7  value='Mostly taragoose'
VIV: eaf_id=a5  value='°Thanks°'
MIC: eaf_id=a8  value="Bes' girl does'n it?"
SHA: eaf_id=a2  value='hm-hmh\xa0yih hmh (.) ˙huhh Best nh hnh Best'


## Gaps and pauses

`gaps(doc)` computes the inter-turn interval between every pair of temporally adjacent
utterances (sorted by start time).  Positive = silence; negative = timing overlap.

`pauses(doc, min_duration=0.2)` filters to notable silences only.

In [23]:
# All inter-turn intervals
for g in gaps(doc):
    print(g)

Gap('SHA'→'VIV', +0.282s [gap])
Gap('VIV'→'MIC', +0.185s [gap])
Gap('MIC'→'VIV', +0.178s [gap])
Gap('VIV'→'MIC', +0.227s [gap])
Gap('MIC'→'VIV', +0.243s [gap])
Gap('VIV'→'MIC', +0.407s [gap])
Gap('MIC'→'SHA', +2.086s [gap])


In [24]:
# Only notable silences (>= 0.2 s)
for g in pauses(doc):
    print(f'{g.duration:.3f}s  {g.before.participant} → {g.after.participant}')
    print(f'  before: {g.before.text!r}')
    print(f'  after:  {g.after.text!r}')

0.282s  SHA → VIV
  before: "ugliest so we gotta make c'ncessions hh ˙hhhheh"
  after:  'n:heyuh'
0.227s  VIV → MIC
  before: 'Move a little, can you? '
  after:  'Mostly taragoose'
0.243s  MIC → VIV
  before: 'Mostly taragoose'
  after:  '°Thanks°'
0.407s  VIV → MIC
  before: '°Thanks°'
  after:  "Bes' girl does'n it?"
2.086s  MIC → SHA
  before: "Bes' girl does'n it?"
  after:  'hm-hmh\xa0yih hmh (.) ˙huhh Best nh hnh Best'


In [25]:
# Filter by duration range
from mumo import gaps

silences     = gaps(doc, min_duration=0)          # gaps only, no overlaps
micro_pauses = gaps(doc, min_duration=0, max_duration=0.2)  # < 0.2 s
long_pauses  = gaps(doc, min_duration=0.5)        # >= 0.5 s
overlaps     = overlaps_by_timing(doc)            # timing-based overlaps only

print(f'{len(silences)} silences, {len(micro_pauses)} micro-pauses, '
      f'{len(long_pauses)} long pauses, {len(overlaps)} timing overlaps')

7 silences, 2 micro-pauses, 1 long pauses, 0 timing overlaps


In [26]:
# Gap objects carry the adjacent utterances and the duration
g = gaps(doc)[0]
print('duration:  ', g.duration)
print('is_overlap:', g.is_overlap)
print('is_silence:', g.is_silence)
print('before:    ', g.before)
print('after:     ', g.after)

duration:   0.28200000000000003
is_overlap: False
is_silence: True
before:     Utterance('SHA', "ugliest so we gotta make c'ncessions hh ")
after:      Utterance('VIV', 'n:heyuh')


## DataFrames

`patterns_df` produces a wide-format DataFrame — one row per pattern instance,
with columns for each slot's text, participant, time, and metric values.

In [27]:
# All patterns across all schemas
result = patterns_df(doc)
result  # dict of {schema_name: DataFrame} when multiple schemas have instances

,pattern_id,schema,note,fpp,fpp_participant,fpp_start,fpp_end,fpp_FPP type,spp,spp_participant,spp_start,spp_end,spp_SPP type,spp_Preferred?
0,1783311410290-5m2k683d4s3g,Adjacency Pair,None,Bes' girl does'n it?,MIC,6.734,7.734,assessment,None,None,None,None,acceptance,None


In [28]:
# Single schema → DataFrame directly
df = patterns_df(doc, schema='Adjacency Pair')
df

,pattern_id,schema,note,fpp,fpp_participant,fpp_start,fpp_end,fpp_FPP type,spp,spp_participant,spp_start,spp_end,spp_SPP type,spp_Preferred?
0,1783311410290-5m2k683d4s3g,Adjacency Pair,None,Bes' girl does'n it?,MIC,6.734,7.734,assessment,None,None,None,None,acceptance,None


In [29]:
print(df.columns.tolist())

['pattern_id', 'schema', 'note', 'fpp', 'fpp_participant', 'fpp_start', 'fpp_end', 'fpp_FPP type', 'spp', 'spp_participant', 'spp_start', 'spp_end', 'spp_SPP type', 'spp_Preferred?']
